In [ ]:
"""
Info:
top 10 najjacih kupaca kupuje od 70k - 15k itema
unique racuna (invoice #) ima oko 100 puta manje nego redaka u tablici
Datumi:Oldest Invoice Date: 2018-01-01 00:00:00
Youngest Invoice Date: 2020-06-30 00:00:00
Oldest Order Date: 1993-04-29 00:00:00 - OVA DVA SU EXTRA SUS
Youngest Order Date: 9999-12-31 00:00:00  - IMA 1000 + o vakvih datuma


"""

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score as sklearn_silhouette_score
from customer_vectorization_functions import function1
from segmentation_functions import my_adjusted_kmeans, merge_kmeans, hierarchical_stratification

In [ ]:
#ucčitavanje podataka
data_path = r"C:\Users\lovro\Desktop\hackatoni\LUMEN_DS.csv"
df = pd.read_csv(
    data_path,
    sep="|",
    quotechar='"',
    encoding="utf-16",)

In [ ]:
df.info()

In [ ]:
a = df["Manufacturing Region"][~df["Manufacturing Region"].isnull()]
np.unique(a).shape

In [ ]:
df = df[~df["Item Code"].isnull()]
codes = df["Item Code"].astype("string")


unique_codes = pd.unique(codes.dropna())
item2idx = {code: int(idx) for idx, code in enumerate(unique_codes)}

# Map to integer ids
df["item_idx"] = codes.map(item2idx)

In [ ]:
df["item_idx"]

In [ ]:
price = df["Invoiced price"]
price_tx = df["Invoiced price (TX)"]

In [ ]:
codes.isnull().sum()

In [ ]:
x = df["Invoiced qty (shipped)"]
mask = x == np.floor(x)
dt = df[~mask]


items_dt = np.unique(dt["Item Code"])
items_df = np.unique(df["Item Code"])

In [ ]:
x[x != np.floor(x)]

In [ ]:
a= (df["Invoiced price"] - df["Invoiced price (TX)"])**2

In [ ]:
print((df["Invoiced price"] <= df["Invoiced price (TX)"]).sum())
print((df["Invoiced price"] > df["Invoiced price (TX)"]).sum())

In [ ]:
df[["Invoiced price", "Invoiced price (TX)"]][a > 0]

In [ ]:
print(a[a <0])

In [ ]:
df[df["Invoice #"] == 16927662]["Order Date"]

In [ ]:
df["Invoice Date"]

In [ ]:
#oldest and youngest InvoiceDate who has format "yyyy-mm-dd" in string format if not in format "yyyy-mm-dd" convert it to that format and then find oldest and youngest date
df["Order Date"] = pd.to_datetime(df["Order Date"], format="%Y-%m-%d", errors='coerce')
oldest_date = df["Order Date"].min()
youngest_date = df["Order Date"].max()
#second youngest date
second_youngest_date = df["Order Date"].nlargest(2).iloc[-1]
print("Oldest Order Date:", oldest_date)
print("Youngest Order Date:", youngest_date)
print("Second Youngest Order Date:", second_youngest_date)

In [ ]:
df["order"]

In [ ]:
import matplotlib.pyplot as plt

df["Order Date"] = pd.to_datetime(df["Order Date"], errors="coerce")

daily_counts = df.groupby(df["Order Date"].dt.date).size()

plt.figure(figsize=(12, 5))
daily_counts.plot()
plt.xlabel("Date")
plt.ylabel("Number of Orders")
plt.title("Orders Over Time")
plt.tight_layout()
plt.show()


In [ ]:
parsed = pd.to_datetime(df["Order Date"], errors="coerce")
invalid_count = parsed.isna().sum()

print("Invalid date values:", invalid_count)

In [ ]:
columns = [""]

In [ ]:
people = df["CustomerID"].unique()
print(f"Total unique customers: {len(people)}")

to_remove = np.zeros((df.shape[0], len(conditions)), dtype=bool)
people_removed = np.zeros((len(conditions),), dtype=int)
for i, person in enumerate(people): # iteriramo po svim ljudima
    
    mask = df["CustomerID"] == person
    person_data = df[mask].copy() 

    for i, uvjet in enumerate(conditions):
        if uvjet(person_data):
            to_remove[mask, i] = True
            people_removed[i] += 1


in conditions we have functions that check for number of invlaid data points for a column
    

print(f"Total customers with more than 10% invalid dates: {number_of_people}")
print(f"Total dates for these customers: {number_of_dates}")
print(f"Average percentage of invalid dates for these customers: {sum_percentages/number_of_people:.2%}")
print(f"Total invalid customers with more than 50 orders: {osobe_s_50}")
print(f"Total valid customers with more than 50 orders: {osobe_valid_50}")
print(f"Percentage of invalid customers with more than 50 orders: {osobe_valid_50:.2%}")

In [ ]:
df["Order Date2"] = pd.to_datetime(df["Order Date"], format="%Y-%m-%d", errors='coerce')

df["Order Date"][df["Order Date2"].isna()]

In [ ]:
#top 10 newest dates in Order Date
top_10_newest_dates = df["Order Date"].nlargest(10000)
print("Top 10 Newest Order Dates:")
print(top_10_newest_dates)

In [ ]:
df.head()

In [ ]:
"""
for i in df.columns:
    if df[i].dtype == "object":
        df[i][df[i].isnull()] = "Unknown"

a,c = np.unique(df["Invoice Line #"], return_counts=True)
sorted_indices = np.argsort(c)[::-1]
sorted_a = a[sorted_indices]
sorted_c = c[sorted_indices]
print(sorted_c[:10])
print(sorted_a[:10])
"""

In [ ]:
#

In [ ]:
#frquency plot of unique values in "CustomerID" column
a,c = np.unique(df["CustomerID"], return_counts=True)
sorted_indices = np.argsort(c)[::-1]
sorted_a = a[sorted_indices]
sorted_c = c[sorted_indices]
plt.figure(figsize=(10, 6))
print(sorted_a[:10])
print(sorted_c[:10])
plt.hist(range(1, 1001), weights=sorted_c[:1000])
"""
plt.xlabel("CustomerID")
plt.ylabel("Frequency")
plt.title("Top 10 CustomerID Frequency")
plt.xticks(rotation=45)
plt.show()"""

In [ ]:
df["Invoice Line #"]

In [ ]:
df.info()

In [ ]:
#all values that apear in Make vs Buy column
print(df["Make vs Buy"].unique())

In [ ]:
#ovdje gledamo koliko svaki covjek/kompanija je napravio narudžbi i po toj mjeri micemo outliere, njih je 1%, njih posebno hendlat

n_orders_per_customer = df.groupby('CustomerID').size().reset_index(name='n_orders')
high_volume_threshold = n_orders_per_customer['n_orders'].quantile(0.99)  # Top 1%
high_volume_customers = n_orders_per_customer[
    n_orders_per_customer['n_orders'] > high_volume_threshold
]['CustomerID'].tolist()

print(f"High-volume customers (>{high_volume_threshold:.0f} orders): {len(high_volume_customers)} ({len(high_volume_customers)/len(n_orders_per_customer)*100:.1f}%)")

# Separate them
df_high_volume = df[df['CustomerID'].isin(high_volume_customers)]
df_regular = df[~df['CustomerID'].isin(high_volume_customers)]

In [ ]:
data_points_imputed_df, data_points_imputed = function1(df_regular)       

In [ ]:
import numpy as np

def metrics(data_points, labels):
    k = labels.max() + 1
    
    # Compute centroids
    centroids = np.vstack([
        data_points[labels == i].mean(axis=0)
        for i in range(k)
    ])
    
    # Silhouette
    silhouette = sklearn_silhouette_score(data_points, labels)
    
    # Mean distance to assigned centroid
    mean_centroid_distance = np.mean(
        np.linalg.norm(data_points - centroids[labels], axis=1)
    )
    
    return silhouette, mean_centroid_distance

In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import seaborn as sns

"""
 tu se računa klasični k means i silouhette score(to nisam ziher kaj je to je chat dao) 
 za raličit broj clustera da se odredi optimalni k
"""

column_names = ['days_diff', 'last_order', 'n_orders', 'avg_price', 
                'avg_ordered_qty', 'avg_delivered_qty',
                'corr_price_ordered', 'corr_price_delivered', 
                'corr_qty', 'gm']


# 2. Scale the data (crucial for k-means!)
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data_points_imputed_df)

# 3. Find optimal number of clusters using elbow method
inertias = []
silhouette_scores = []
K_range = range(2, 30)

for k in K_range:
    """
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(data_scaled)
    silhouette_score, inertia = metrics(data_scaled, kmeans.labels_)
    """
    #ovo je adjusted k means koji uvijek daje clustere koji imaju barem 100 članova
    #final_labels = my_adjusted_kmeans(data_scaled,k, min_size=100)
    #final_labels = merge_kmeans(data_scaled, k, min_size=100)
    final_labels = hierarchical_stratification(data_scaled, k, min_size=100)
    
    silhouette_score, inertia = metrics(data_scaled, final_labels)
    
    inertias.append(inertia)
    silhouette_scores.append(silhouette_score)

# Plot elbow curve
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Elbow method plot
axes[0].plot(K_range, inertias, 'bo-')
axes[0].set_xlabel('Number of clusters (k)')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method for Optimal k')
axes[0].grid(True)

# Silhouette score plot
axes[1].plot(K_range, silhouette_scores, 'ro-')
axes[1].set_xlabel('Number of clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score for Optimal k')
axes[1].grid(True)

plt.tight_layout()
plt.show()

# Print silhouette scores
for k, score in zip(K_range, silhouette_scores):
    print(f"k={k}: Silhouette Score = {score:.4f}")

# 4. Choose optimal k (let's say based on silhouette score)


In [ ]:
data_scaled.shape

In [ ]:
# odabrao sam 11 tu sad dalje ide neka analiza i vizualizacija
optimal_k = 12
print(f"\nOptimal number of clusters: {optimal_k}")

print(data_scaled.shape)

# ovo je klasicni kmeans
"""
final_kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
cluster_labels = final_kmeans.fit_predict(data_scaled)
"""
#ovo je adjusted k means koji uvijek daje clustere koji imaju barem 100 članova
#cluster_labels =  my_adjusted_kmeans(data_scaled, optimal_k, min_size=100)
#cluster_labels = merge_kmeans(data_scaled, optimal_k, min_size=100)
cluster_labels = hierarchical_stratification(data_scaled, optimal_k, min_size=100)

# Add cluster labels to your dataframe
labeled_data_points_imputed_df = data_points_imputed_df.copy()
labeled_data_points_imputed_df['Cluster'] = cluster_labels

# 6. Analyze cluster characteristics
print("\n=== Cluster Analysis ===")
print("\nCluster sizes:")
print(labeled_data_points_imputed_df['Cluster'].value_counts().sort_index())

# Calculate mean values for each cluster
cluster_means = labeled_data_points_imputed_df.groupby('Cluster').mean()
print("\nCluster means (scaled back to original units):")
print(cluster_means)

# 7. Visualize cluster profiles
# Create a heatmap of cluster characteristics
plt.figure(figsize=(12, 6))
sns.heatmap(cluster_means.T, annot=True, cmap='coolwarm', center=0, 
            fmt='.2f', cbar_kws={'label': 'Mean Value'})
plt.title('Cluster Profiles - Mean Values')
plt.ylabel('Features')
plt.xlabel('Cluster')
plt.tight_layout()
plt.show()

# 8. Visualize clusters using PCA for 2D visualization
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
data_pca = pca.fit_transform(data_scaled)

plt.figure(figsize=(10, 8))
scatter = plt.scatter(data_pca[:, 0], data_pca[:, 1], 
                     c=cluster_labels, cmap='viridis', alpha=0.6)
plt.colorbar(scatter, label='Cluster')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)')
plt.title('Customer Segments Visualization (PCA)')
plt.grid(True, alpha=0.3)
plt.show()

# 9. Analyze feature importance for clustering
# Look at cluster centers
cluster_centers = scaler.inverse_transform(final_kmeans.cluster_centers_)
cluster_centers_df = pd.DataFrame(cluster_centers, columns=column_names)
print("\nCluster Centers (original units):")
print(cluster_centers_df)


# 11. Optional: Characterize each cluster
print("\n=== Cluster Characterization ===")
for cluster in range(optimal_k):
    print(f"\nCluster {cluster} (n={sum(cluster_labels==cluster)} customers):")
    cluster_data = labeled_data_points_imputed_df[labeled_data_points_imputed_df['Cluster'] == cluster]
    
    # Print key characteristics
    print(f"  Average days between orders: {cluster_data['days_diff'].mean():.1f}")
    print(f"  Average order value: ${cluster_data['avg_price'].mean():.2f}")
    print(f"  Average order quantity: {cluster_data['avg_ordered_qty'].mean():.1f}")
    print(f"  Average GM%: {cluster_data['gm'].mean():.1f}%")
    print(f"  Price-Order correlation: {cluster_data['corr_price_ordered'].mean():.3f}")

In [ ]:
"""
postoje invoice price 0 narudzbe, onda gm% bude nan i nemogu korelaciju gledati
ponekad je inovice price 0, a product cost nije, mogao bih onda gm% postaviti na -max
order qty i deliver qty se razlikuju
invoice dateovi neki su invalidni


nisam zadovoljan sve skupa jer postoje klasteri s po jednom osobom, možda su to ovi izdvojeni iz skupine na slici
"""